# 07 Region Discovery

**Purpose.** Test whether the severity signal is localised on the mass axis or distributed.

**Thesis reference.** §5 Localization

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [7]:
PROFILE_DIR = "profiles"
LABELS      = "labels/labels_severity_nondiabetic.json"
OUT_FILE    = "regions_severity_nondiabetic.json"
N_REGIONS   = [10, 25, 50, 100]
N_REPEATS   = 5

ARGS = ["--profiles", PROFILE_DIR,
        "--labels", LABELS,
        "--out", OUT_FILE,
        "--n-regions", *[str(k) for k in N_REGIONS],
        "--n-repeats", str(N_REPEATS)]

## Code

In [8]:
import argparse
import json
import warnings
from pathlib import Path

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
ISOTOPE_SPACING = 1.0033548

# Normalisation
def normalise_profiles(F):
    """
    Total-ion-current normalisation followed by log compression.

    TIC normalisation makes patients comparable despite differences in sample
    loading and ionisation efficiency. This is the standard choice in MS and
    is preferable to the per-image min-max scaling used in the image pipeline,
    which discards absolute intensity entirely.
    """
    tic = F.sum(axis=1, keepdims=True)
    tic[tic == 0] = 1.0
    return np.log1p(F / tic * 1e6)

# Fold-safe region selection
class RegionSelector(BaseEstimator, TransformerMixin):
    """
    Selects the top-k discriminative m/z windows.

    fit() computes a univariate discriminative score per bin using ONLY the
    data it is given. Inside a cross-validation loop that is the training
    fold, so no test information can influence which regions are chosen.

    Score is |AUC - 0.5| per bin, which is rank based and therefore robust to
    the heavy-tailed intensity distributions typical of MS data.
    """

    def __init__(self, mz_centres, n_regions=40, half_width_da=0.08,
                 min_separation_da=2.0, mode="region"):
        """
        half_width_da defaults to 0.08 Da for two measured reasons.

        Lower bound --- mass accuracy. Step 8 measured a per-patient m/z shift
        of about +/-0.02 Da (roughly one bin at 0.02 Da resolution), random
        rather than trending with acquisition order. A window narrower than
        that would sometimes miss the peak it is meant to capture.

        Upper bound --- isotope separation. Isotopologues are separated by
        1.0034/z Da, so adjacent isotope windows overlap unless
        half_width < 0.5017/z, which is 0.167 Da at charge state 3. The
        earlier default of 0.5 Da produced isotope windows that overlapped
        almost completely at z = 2 and z = 3, making those features duplicates
        of the monoisotopic window rather than independent measurements.

        0.08 Da sits comfortably inside both bounds.
        """
        self.mz_centres = mz_centres
        self.n_regions = n_regions
        self.half_width_da = half_width_da
        self.min_separation_da = min_separation_da
        self.mode = mode

    # scoring
    @staticmethod
    def _bin_auc(X, y):
        """Vectorised per-column AUC via rank statistics."""
        n_pos = int(y.sum())
        n_neg = len(y) - n_pos
        if n_pos == 0 or n_neg == 0:
            return np.full(X.shape[1], 0.5)
        order = np.argsort(X, axis=0)
        ranks = np.empty_like(order, dtype=np.float64)
        rows = np.arange(X.shape[0])[:, None]
        ranks[order, np.arange(X.shape[1])[None, :]] = rows + 1
        pos_rank_sum = ranks[y == 1].sum(axis=0)
        return (pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)

    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        auc = self._bin_auc(X, y)
        score = np.abs(auc - 0.5)

        # Greedy peak picking with a minimum m/z separation, so the selected
        # regions are distinct molecular features rather than 40 adjacent
        # bins of the same peak.
        res = self.mz_centres[1] - self.mz_centres[0]
        sep_bins = max(1, int(self.min_separation_da / res))

        order = np.argsort(score)[::-1]
        chosen = []
        taken = np.zeros(len(score), dtype=bool)
        for idx in order:
            if len(chosen) >= self.n_regions:
                break
            if taken[idx]:
                continue
            chosen.append(idx)
            lo = max(0, idx - sep_bins)
            hi = min(len(score), idx + sep_bins + 1)
            taken[lo:hi] = True

        self.centres_ = np.sort(np.array(chosen))
        self.centre_mz_ = self.mz_centres[self.centres_]
        self.scores_ = score[self.centres_]
        self.auc_ = auc[self.centres_]
        self.half_width_bins_ = max(1, int(self.half_width_da / res))
        self.resolution_ = res
        return self

    # feature construction
    def _window(self, X, centre):
        lo = max(0, centre - self.half_width_bins_)
        hi = min(X.shape[1], centre + self.half_width_bins_ + 1)
        return X[:, lo:hi]

    def _region_features(self, X):
        feats = [self._window(X, c).sum(axis=1) for c in self.centres_]
        return np.column_stack(feats)

    def _isotope_features(self, X):
        """
        For each region, describe the isotopic envelope rather than just its
        total intensity.

        For charge states z = 1, 2, 3 the expected spacing between
        isotopologues is ISOTOPE_SPACING / z. The ratio of intensity at the
        monoisotopic position to intensity one isotope step away is
        informative about elemental composition --- in particular carbon
        count, which for lipids tracks chain length.

        This is the "fixed distance" filter: a matched comparison at a
        physically determined offset, which is only possible because the
        profile is binned finely enough to resolve it.
        """
        feats = []
        for c in self.centres_:
            base = self._window(X, c).sum(axis=1)
            feats.append(base)
            for z in (1, 2, 3):
                shift = int(round((ISOTOPE_SPACING / z) / self.resolution_))
                c2 = c + shift
                if c2 + self.half_width_bins_ >= X.shape[1]:
                    feats.append(np.zeros(len(X)))
                    continue
                nxt = self._window(X, c2).sum(axis=1)
                feats.append(nxt)
                feats.append(nxt / (base + 1e-9))   # isotope ratio
        return np.column_stack(feats)

    def transform(self, X):
        if self.mode == "isotope":
            return self._isotope_features(X)
        return self._region_features(X)

class CoarseBinner(BaseEstimator, TransformerMixin):
    """Whole-spectrum baseline: rebin the fine profile down to n_bins."""

    def __init__(self, n_bins=256):
        self.n_bins = n_bins

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        n = X.shape[1] // self.n_bins * self.n_bins
        return X[:, :n].reshape(len(X), self.n_bins, -1).sum(axis=2)

# Pipelines
def make_pipeline(strategy, mz_centres, n_regions, C=1.0):
    clf = LogisticRegression(C=C, max_iter=5000, class_weight="balanced")
    if strategy == "binned":
        steps = [("bin", CoarseBinner(256))]
    else:
        steps = [("sel", RegionSelector(mz_centres, n_regions=n_regions,
                                        mode=strategy))]
    steps += [("scale", StandardScaler()), ("clf", clf)]
    return Pipeline(steps)

def cv_auc(pipe, X, y, n_splits=5, n_repeats=5, seed=RANDOM_STATE):
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                   random_state=seed)
    aucs = []
    for tr, te in rskf.split(X, y):
        p = pipe.fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        if len(np.unique(y[te])) > 1:
            aucs.append(roc_auc_score(y[te], p))
    return float(np.mean(aucs)), float(np.std(aucs))

# Main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profiles", required=True)
    ap.add_argument("--labels", required=True)
    ap.add_argument("--out", default="region_results.json")
    ap.add_argument("--n-regions", nargs="+", type=int, default=[10, 25, 50])
    ap.add_argument("--n-repeats", type=int, default=5)
    ap.add_argument("--permutation-test", action="store_true", default=True)
    args = ap.parse_args()

    prof = Path(args.profiles)
    F = np.load(prof / "mz_profiles.npy")
    mz_centres = np.load(prof / "mz_centres.npy")
    with open(prof / "profile_pids.json") as fh:
        pids = json.load(fh)
    with open(args.labels) as fh:
        obj = json.load(fh)
    labels = obj.get("labels", obj)

    keep = [i for i, p in enumerate(pids) if p in labels]
    X = normalise_profiles(F[keep])
    y = np.array([labels[pids[i]] for i in keep], dtype=int)
    kept_pids = [pids[i] for i in keep]

    print(f"Patients with profile + label: {len(y)} "
          f"({int(y.sum())} positive / {int((1 - y).sum())} negative)")
    print(f"Fine m/z bins: {X.shape[1]:,}\n")

    # held-out test set, touched once
    idx = np.arange(len(y))
    idx_dev, idx_test = train_test_split(idx, test_size=0.2, stratify=y,
                                         random_state=RANDOM_STATE)
    X_dev, y_dev = X[idx_dev], y[idx_dev]
    X_test, y_test = X[idx_test], y[idx_test]
    print(f"dev {len(y_dev)} | test {len(y_test)}\n")

    results = {}
    for strategy in ["binned", "region", "isotope"]:
        n_list = [0] if strategy == "binned" else args.n_regions
        for n_regions in n_list:
            best = None
            for C in [0.01, 0.1, 1.0]:
                pipe = make_pipeline(strategy, mz_centres, n_regions, C)
                m, s = cv_auc(pipe, X_dev, y_dev, n_repeats=args.n_repeats)
                if best is None or m > best[0]:
                    best = (m, s, C)
            m, s, C = best

            pipe = make_pipeline(strategy, mz_centres, n_regions, C)
            pipe.fit(X_dev, y_dev)
            probs = pipe.predict_proba(X_test)[:, 1]
            t_auc = roc_auc_score(y_test, probs)

            key = f"{strategy}_k{n_regions}"
            results[key] = {
                "strategy": strategy,
                "n_regions": n_regions,
                "C": C,
                "cv_auc_mean": m,
                "cv_auc_std": s,
                "test_auc": float(t_auc),
            }

            print(f"{key:16s} CV {m:.4f} +/- {s:.4f}   TEST {t_auc:.4f}")

            if strategy == "region" and "sel" in pipe.named_steps:
                sel = pipe.named_steps["sel"]
                top = np.argsort(sel.scores_)[::-1][:15]
                results[key]["top_regions"] = [
                    {"mz": float(sel.centre_mz_[i]),
                     "auc": float(sel.auc_[i])} for i in top
                ]

    # permutation control
    if args.permutation_test:
        print("\nPermutation control (labels shuffled; AUC should be ~0.5)")
        rng = np.random.default_rng(RANDOM_STATE)
        perm_aucs = []
        for r in range(5):
            y_perm = rng.permutation(y_dev)
            pipe = make_pipeline("region", mz_centres,
                                 max(args.n_regions), 1.0)
            m, _ = cv_auc(pipe, X_dev, y_perm, n_repeats=1, seed=r)
            perm_aucs.append(m)
            print(f"  run {r + 1}: {m:.4f}")
        results["permutation_control"] = {
            "mean": float(np.mean(perm_aucs)),
            "runs": [float(a) for a in perm_aucs],
        }
        pm = np.mean(perm_aucs)
        print(f"  mean {pm:.4f}"
              + ("  <-- OK" if abs(pm - 0.5) < 0.08
                 else "  <-- WARNING: possible leakage"))

    # final region report on the dev set
    sel = RegionSelector(mz_centres, n_regions=max(args.n_regions),
                         mode="region").fit(X_dev, y_dev)
    order = np.argsort(sel.scores_)[::-1]
    print("\nMost discriminative m/z windows (fitted on dev set only)")
    print(f"{'m/z':>10}  {'AUC':>7}  {'|AUC-0.5|':>9}")
    for i in order[:20]:
        print(f"{sel.centre_mz_[i]:10.3f}  {sel.auc_[i]:7.4f}  "
              f"{sel.scores_[i]:9.4f}")

    results["reported_regions"] = [
        {"mz": float(sel.centre_mz_[i]), "auc": float(sel.auc_[i])}
        for i in order
    ]
    results["cohort"] = {"n": len(y), "n_positive": int(y.sum()),
                         "pids": kept_pids}

    with open(args.out, "w") as fh:
        json.dump(results, fh, indent=2)
    print(f"\nWritten to {args.out}")

## Run

In [9]:
import sys
sys.argv = ["05_region_discovery.py"] + ARGS
main()

Patients with profile + label: 359 (66 positive / 293 negative)
Fine m/z bins: 80,000

dev 287 | test 72

binned_k0        CV 0.6398 +/- 0.0656   TEST 0.5489
region_k10       CV 0.5208 +/- 0.0920   TEST 0.3377
region_k25       CV 0.5528 +/- 0.0733   TEST 0.4602
region_k50       CV 0.5939 +/- 0.0706   TEST 0.5072
region_k100      CV 0.6133 +/- 0.0959   TEST 0.5202
isotope_k10      CV 0.5644 +/- 0.0893   TEST 0.4681
isotope_k25      CV 0.5614 +/- 0.1030   TEST 0.4798
isotope_k50      CV 0.5829 +/- 0.0920   TEST 0.5059
isotope_k100     CV 0.5968 +/- 0.0901   TEST 0.6128

Permutation control (labels shuffled; AUC should be ~0.5)
  run 1: 0.5573
  run 2: 0.4834
  run 3: 0.5376
  run 4: 0.3876
  run 5: 0.5585
  mean 0.5049  <-- OK

Most discriminative m/z windows (fitted on dev set only)
       m/z      AUC  |AUC-0.5|
  1551.310   0.6760     0.1760
  1485.890   0.6757     0.1757
   579.410   0.3297     0.1703
  1616.370   0.6691     0.1691
   321.630   0.6687     0.1687
  1494.510   0.6686  